# Data preprocessing

Embeddings use **SaProt** alone: a structure-aware PLM that fuses amino-acid identity and Foldseek 3Di structure tokens into one vocabulary, needing only the real WT PDB already on disk (no folding, no OOM risk). See `docs/02_data_preprocessing.md`.

ESM-2 sequence embeddings concatenated alongside SaProt (`structure_and_sequence`) were tried and dropped: at this sample count (~600-900 training rows) the larger concatenated input (5120-dim vs. 2560-dim) was too big for a light head and gave no measurable improvement over SaProt alone (`structure_only`).

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.processed_io import load_mutation_records
from shared.constants import SPLIT_FILES, SplitName

records = load_mutation_records()
print(f"1211 raw AB/AG rows -> 154 homology-duplicates discarded -> {len(records)} final samples")

pd.DataFrame([
    {"split_scheme": split.value, "subset": subset, "n": len(pd.read_csv(SPLIT_FILES[split][subset]))}
    for split in SplitName
    for subset in ("train", "val")
]).pivot(index="split_scheme", columns="subset", values="n")

1211 raw AB/AG rows -> 154 homology-duplicates discarded -> 1057 final samples


subset,train,val
split_scheme,,
held_out_pdb,872,185
same_pdb_allowed,855,202


Extraction is fast: ~0.01-0.04s/PDB for the Foldseek structure tokenizer, ~0.04-0.07s/chain for the SaProt forward pass itself.